# 07 · Positive values and prediction intervals

Two task types for numbers beyond plain regression:

| Task | Use it when | Output |
|---|---|---|
| `positive_regression` | the value can never be negative (days until the next visit, quantities) | one value, always > 0 |
| `quantile` | you want a **range**, not just one number | the median plus the bounds of each `coverage` level |

`coverage` can be any level, or several at once: `coverage=0.95` for a wide "almost always" range,
`coverage=0.5` for a narrow "typical" range, or `coverage=(0.5, 0.95)` for both.

> A narrower interval is **not** a more precise model: a 50% interval misses the true value half of the
> time by design. To get narrower intervals at the same coverage, improve the model or the features.

## The example data

Every cookbook uses the same small synthetic shop: each row is one purchase with an `item`,
a `price` and a `weekday`. Shoppers buy the five items in a fixed cycle
(`apple → bread → cheese → dates → eggs → apple ...`), starting at a random item, so the next
item is learnable and we can check the model gets it right.

Here every purchase also has a `gap`: days since the shopper's previous purchase. It is always
positive and depends on the item (cheese is bought rarely, bread often), with a lot of random
variation. This notebook uses 1,000 shoppers so the measured coverage is precise enough to read.

In [1]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

import math
import torch
from torch.utils.data import DataLoader
from next_sequence import (CoreDataset, CustomCollate, Feature, NextSequence, NextSequenceConfig,
                           create_bow, make_collate, targets)

events = make_events(n_users=1000)        # more shoppers: coverage is measured more precisely
rng = random.Random(1)
TYPICAL_GAP = {"apple": 2.0, "bread": 1.0, "cheese": 7.0, "dates": 4.0, "eggs": 3.0}
events["gap"] = [round(TYPICAL_GAP[i] * rng.expovariate(1.0) + 0.1, 2) for i in events["item"]]

users = (events.sort_values(["user_id", "t"]).groupby("user_id").agg(list)
               .reset_index().drop(columns="t").to_dict(orient="records"))

def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]      # target: 3rd-to-last event
val_raw = [cut(u, -1) for u in users]        # target: 2nd-to-last event
test_raw = [cut(u, None) for u in users]     # target: last event

events.head(6)

,user_id,t,item,price,weekday,gap
0,0,0,dates,102.41,0,0.68
1,0,1,eggs,57.11,4,5.74
2,0,2,apple,19.94,6,2.99
3,0,3,bread,47.97,3,0.39
4,0,4,cheese,174.89,1,4.89
5,0,5,dates,95.09,2,2.49


## Preprocessing and targets

- `item`: next item (classification), as in the other cookbooks.
- `gap`: **positive regression**. Scale by **dividing only** (`NextValue(scale=...)`, no `shift`), so
  targets stay >= 0; a positive task rejects negative targets with a clear message.
- `price`: **quantile** with `coverage=(0.5, 0.95)`. The target is `LogRatio` (relative change), so the
  decoded bounds are always positive and can be asymmetric, which suits prices.

Inputs use `(scaled, raw)` steps: the model sees scaled values, the targets use real ones.

In [2]:
bow = create_bow(u["item"] for u in train_raw)

def stats(values):
    mean = sum(values) / len(values)
    return mean, (sum((v - mean) ** 2 for v in values) / len(values)) ** 0.5

log_price_mean, log_price_std = stats([math.log(p) for u in train_raw for p in u["price"]])
gap_mean, gap_std = stats([g for u in train_raw for g in u["gap"]])

raw = lambda step: step[1]
gap_target = targets.NextValue(scale=gap_std, value=raw)                      # divide only: stays >= 0
price_target = targets.LogRatio(scale=targets.target_std([u["price"] for u in train_raw], targets.LogRatio()),
                                value=raw)

def transform(samples):
    return [{"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
             "price": [((math.log(p) - log_price_mean) / log_price_std, p) for p in u["price"]],
             "gap": [((g - gap_mean) / gap_std, g) for g in u["gap"]],
             "weekday": [d + 1 for d in u["weekday"]]} for u in samples]

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)

features = [
    Feature("item", "classification", vocab_size=len(bow)),
    Feature("price", "quantile", is_linear=True, coverage=(0.5, 0.95)),
    Feature("gap", "positive_regression", is_linear=True),
    Feature("weekday", "passthrough", vocab_size=8),
]
funcs = {
    "item": make_collate("long"),
    "price": make_collate("float", x=lambda s: s[0], y=price_target),
    "gap": make_collate("float", x=lambda s: s[0], y=gap_target),
    "weekday": make_collate("long"),
}
config = NextSequenceConfig(features=features, max_len=16, emb_dim=32, attn_heads=2)

def loader(samples, inference=False, shuffle=False):
    collate = CustomCollate.from_config(config, "item", funcs, inference=inference)
    return DataLoader(CoreDataset(samples), batch_size=32, shuffle=shuffle, collate_fn=collate)

print("price quantiles predicted:", features[1].quantiles)

price quantiles predicted: (0.025, 0.25, 0.5, 0.75, 0.975)


## Train

In [3]:
model = NextSequence(config, epochs=25, lr=3e-3, seed=0, verbose=False)
model.fit(loader(train, shuffle=True), loader(val))
pd.DataFrame(model.history_).set_index("epoch").iloc[[0, 9, 19, 24]].round(4)

,train_loss,train_item,train_price,train_gap,val_loss,val_item,val_price,val_gap
epoch,,,,,,,,
1,2.4948,1.6509,0.2591,1.1990,1.7843,1.1700,0.1819,0.9368
10,-0.1670,0.0033,0.0271,0.8333,-0.2131,0.0016,0.0226,0.8342
20,-0.6894,0.0016,0.0239,0.5695,-0.4420,0.0010,0.0219,1.0207
25,-0.7368,0.0017,0.0236,0.5286,-0.4549,0.0010,0.0218,1.0238


## Read the intervals with `interval`

`predict` returns the quantile columns from the lowest up; `model.interval(out, name, coverage)` picks
the right ones and returns `lower`, `median` and `upper`. They are still in the target's scale, so the
target strategy's `decode` turns them into prices (the last price the model saw is `price[-2]`).

In [4]:
out = model.predict(loader(test))
print("price output:", tuple(out["price"].shape), "| gap output:", tuple(out["gap"].shape))

last_price = [u["price"][-2] for u in test_raw]
wide = model.interval(out, "price", coverage=0.95)
typical = model.interval(out, "price", coverage=0.5)

def to_price(t):
    return price_target.decode(t, last=last_price)

pd.DataFrame({
    "item": [u["item"][-1] for u in test_raw[:6]],
    "true price": [u["price"][-1] for u in test_raw[:6]],
    "95% low": to_price(wide.lower)[:6].double().numpy().round(1),
    "50% low": to_price(typical.lower)[:6].double().numpy().round(1),
    "median": to_price(wide.median)[:6].double().numpy().round(1),
    "50% high": to_price(typical.upper)[:6].double().numpy().round(1),
    "95% high": to_price(wide.upper)[:6].double().numpy().round(1),
})

price output: (1000, 5) | gap output: (1000, 1)


,item,true price,95% low,50% low,median,50% high,95% high
0,eggs,57.72,49.9,56.7,60.1,63.2,70.7
1,bread,40.51,42.3,47.6,50.7,53.4,61.3
2,dates,102.13,68.1,76.3,80.9,85.4,93.1
3,cheese,173.41,162.5,184.6,195.4,211.0,252.5
4,dates,87.02,90.3,100.9,108.0,118.0,137.9
5,eggs,61.04,50.5,56.3,59.7,62.7,70.8


## Positive regression: `gap` is never negative

Decoding is `pred * scale` (no shift), so predictions stay positive in days as well. The gap is mostly
random here, so the model can only learn the typical gap of the next item: expect a modest improvement
over always predicting the average, not a precise forecast.

In [5]:
pred_gap = gap_target.decode(out["gap"])
true_gap = torch.tensor([u["gap"][-1] for u in test_raw])
print(f"lowest predicted gap: {pred_gap.min():.3f} days | MAE: {(pred_gap - true_gap).abs().mean():.2f} days "
      f"(predicting the average gap: {(true_gap - gap_mean).abs().mean():.2f})")
pd.DataFrame({"next item": [u["item"][-1] for u in test_raw[:6]],
              "true gap": true_gap[:6].double().numpy().round(2),
              "predicted gap": pred_gap[:6].double().numpy().round(2)})

lowest predicted gap: 0.640 days | MAE: 2.73 days (predicting the average gap: 3.01)


,next item,true gap,predicted gap
0,eggs,1.80,2.69
1,bread,0.58,1.00
2,dates,1.16,4.21
3,cheese,7.30,9.62
4,dates,5.07,12.39
5,eggs,0.21,2.08


## Do the intervals keep their promise? `interval_coverage`

A 95% interval should contain about 95% of the true values on data the model hasn't seen.
`interval_coverage` measures it (`observed`) together with the average width (`mean_width`, in the
target's scale, here the scaled log ratio).

Coverage measured on a finite sample is itself noisy: with 1,000 samples, an interval that truly
covers 95% will show anywhere from about 93.6% to 96.4%.

In [6]:
pd.DataFrame(model.interval_coverage(loader(test))).round(3)

,feature,coverage,observed,mean_width
0,price,0.50,0.463,0.122
1,price,0.95,0.932,0.372


## Calibrate to hit the target coverage

`calibrate` (conformalized quantile regression) measures how far true values fall outside each
interval on held-out data and widens or narrows the interval to match its coverage. Use data the
model was not trained on: here the validation split. The adjustment is applied by `predict`, saved
by `save`, and cleared when the model is trained again.

For the strictest guarantee, calibrate on a split used for nothing else: the validation split also
picked the best epoch, which makes the model look slightly better on it than on new data.

In [7]:
offsets = model.calibrate(loader(val))
print("offsets (in target scale):", {name: {c: round(q, 4) for c, q in v.items()} for name, v in offsets.items()})

before = pd.DataFrame(model.interval_coverage(loader(test), calibrated=False)).assign(calibrated=False)
after = pd.DataFrame(model.interval_coverage(loader(test))).assign(calibrated=True)
pd.concat([before, after]).round(3)

offsets (in target scale): {'price': {0.5: 0.0011, 0.95: 0.0053}}


,feature,coverage,observed,mean_width,calibrated
0,price,0.50,0.463,0.122,False
1,price,0.95,0.932,0.372,False
0,price,0.50,0.471,0.124,True
1,price,0.95,0.936,0.383,True


## Predict a range for the future

With an inference loader the model sees the full history, so the last price is `price[-1]`.

In [8]:
future = model.predict(loader(test, inference=True))
now = [u["price"][-1] for u in test_raw]
iv = model.interval(future, "price", coverage=0.95)
pd.DataFrame({
    "last item": [u["item"][-1] for u in test_raw[:5]],
    "current price": now[:5],
    "next price, 95% range": [f"{lo:.1f} – {hi:.1f}" for lo, hi in zip(
        price_target.decode(iv.lower, last=now)[:5].tolist(), price_target.decode(iv.upper, last=now)[:5].tolist())],
    "next gap (days)": gap_target.decode(future["gap"])[:5].double().numpy().round(2),
})

,last item,current price,"next price, 95% range",next gap (days)
0,eggs,57.72,15.9 – 22.5,3.92
1,bread,40.51,147.5 – 197.7,7.13
2,dates,102.13,49.1 – 72.8,2.11
3,cheese,173.41,71.7 – 103.5,3.20
4,dates,87.02,42.2 – 60.4,4.39


## Save and load keep the calibration

In [9]:
model.save("outputs/intervals.pt", metadata={"bow": bow})
restored = NextSequence.load("outputs/intervals.pt")
same = torch.allclose(restored.predict(loader(test))["price"], model.predict(loader(test))["price"], atol=1e-5)
print("calibration restored:", restored.calibration_ == model.calibration_, "| same predictions:", same)

calibration restored: True | same predictions: True


## Summary

- `positive_regression`: scale targets by dividing only; outputs are always > 0.
- `quantile` with `coverage=(...)`: any levels, several at once, bounds never cross.
- `model.interval(out, name, coverage)` → `lower`, `median`, `upper`; decode them with your target strategy.
- `model.interval_coverage(loader)` checks the promise; `model.calibrate(val_loader)` fixes it.
- Narrow intervals make a weaker promise; better models make narrower intervals at the same coverage.